# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 3f808024-0944-431d-a865-c538c5… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 556bb81f-3fb2-4f6f-bea5-739f0c… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-02 ┆ claude-hai ┆ 565.0      ┆ 1140.0    ┆ 16651653  ┆ 8245961   ┆ 0.049327  ┆ 46.305166 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## Kiểm tra bổ sung cho Gold

Notebook gốc chỉ assert `Silver < Bronze` và `≥ 7 dates`. Cell dưới kiểm tra thêm các điều kiện
còn lại của rubric/checkpoint (p50 ≤ p95, `cost_usd > 0`, `error_rate ∈ [0, 1]`) và in đủ bảng Gold.
Cell tự đọc lại 3 bảng từ đĩa nên có thể chạy riêng, không phụ thuộc biến của các cell trước.

In [6]:
import _setup  # noqa: F401
from pathlib import Path
import polars as pl
from deltalake import DeltaTable
from lakehouse import path

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
gold_df  = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table()).sort(["date", "model"])
n_dates, n_models = gold_df["date"].n_unique(), gold_df["model"].n_unique()

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(gold_df.select("date", "model", "p50_latency_ms", "p95_latency_ms", "error_rate", "cost_usd"))

print(f"\nBronze {bronze_n:,} → Silver {silver_n:,}  (dropped {bronze_n - silver_n:,})")
print(f"Gold: {n_dates} dates × {n_models} models = {gold_df.height} rows")

gold_checks = {
    "3 tables on disk (_delta_log)": all(Path(p, "_delta_log").exists() for p in (BRONZE, SILVER, GOLD)),
    "Silver < Bronze":               silver_n < bronze_n,
    "≥ 7 dates × 3 models":          n_dates >= 7 and n_models >= 3 and gold_df.height == n_dates * n_models,
    "p50 ≤ p95 everywhere":          bool((gold_df["p50_latency_ms"] <= gold_df["p95_latency_ms"]).all()),
    "cost_usd > 0":                  bool((gold_df["cost_usd"] > 0).all()),
    "error_rate in [0, 1]":          bool(gold_df["error_rate"].is_between(0, 1).all()),
    "error_rate not all zero":       bool((gold_df["error_rate"] > 0).any()),
}
for k, v in gold_checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(gold_checks.values()), "Gold check failed — see FAIL rows above"

shape: (24, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬────────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd   │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---        │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ f64        ┆ f64        │
╞════════════╪═══════════════════╪════════════════╪════════════════╪════════════╪════════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 567.0          ┆ 1121.0         ┆ 0.04936    ┆ 33.178525  │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 3045.0         ┆ 6035.65        ┆ 0.049735   ┆ 200.248575 │
│ 2026-04-01 ┆ claude-sonnet-4-6 ┆ 1381.0         ┆ 2752.0         ┆ 0.050408   ┆ 244.145982 │
│ 2026-04-02 ┆ claude-haiku-4-5  ┆ 565.0          ┆ 1140.0         ┆ 0.049327   ┆ 46.305166  │
│ 2026-04-02 ┆ claude-opus-4-7   ┆ 3004.0         ┆ 5972.0         ┆ 0.052803   ┆ 291.28575  │
│ 2026-04-02 ┆ claude-sonnet-4-6 ┆ 

## Giải thích kết quả NB4

**Số liệu chính**

| Tầng | Bảng | Kết quả |
|---|---|---|
| Bronze | `bronze/llm_calls_raw` | 200.000 dòng thô (`request_id`, `ts`, `raw_json`) |
| Silver | `silver/llm_calls` | 190.052 dòng, partition theo `date` (8 thư mục `date=2026-04-01` … `date=2026-04-08`) |
| Gold | `gold/llm_daily_metrics` | 24 dòng = **8 ngày × 3 model** (haiku-4-5, sonnet-4-6, opus-4-7) |

**1. Bronze — giữ nguyên dữ liệu thô**
Bronze lưu payload đúng như nhận được (`raw_json` dạng chuỗi), kể cả bản ghi trùng do client retry.
Không sửa gì ở tầng này để luôn có thể xử lý lại (replay) Silver/Gold khi logic thay đổi hoặc phát hiện bug.

**2. Silver — parse, chuẩn hóa kiểu, dedup**
DuckDB parse JSON thành cột có kiểu (`model`, `user_id`, `prompt_tokens`, `completion_tokens`,
`latency_ms`, `status`), thêm cột `date` từ `ts`, rồi dedup bằng
`ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)` và giữ `rn = 1` (bản ghi sớm nhất).
Kết quả: 200.000 → 190.052, **loại 9.948 dòng**. Số `request_id` khác nhau trong Bronze cũng đúng
bằng 190.052, nên toàn bộ phần chênh lệch là bản ghi retry trùng; bộ lọc `model IS NOT NULL` không loại thêm dòng nào.
Nếu không dedup, token và chi phí ở Gold sẽ bị tính dư khoảng 5%.

**3. Gold — tổng hợp phục vụ câu hỏi nghiệp vụ**
Gom theo `(date, model)`: p50/p95 latency (`QUANTILE_CONT`), tổng token, `error_rate`
(tỉ lệ `status <> 'ok'`, gồm `error` và `rate_limited`) và `cost_usd` = token × giá minh họa / 1M.
Kiểm tra bổ sung ở cell phía trên cho thấy:
- p50 ≤ p95 ở cả 24 dòng; latency tăng theo cỡ model: haiku p50 ≈ 560 ms, sonnet ≈ 1.390 ms, opus ≈ 3.000 ms.
- `cost_usd > 0` ở mọi dòng; tổng ≈ 4.755 USD trong 8 ngày. Sonnet tốn nhất mỗi ngày (≈ 345 USD) vì
  lượng token lớn nhất, dù giá/token thấp hơn opus.
- `error_rate` nằm trong khoảng 0,042–0,062, tức khoảng 5% lỗi.
- Ngày 2026-04-01 và 2026-04-08 có ít request hơn (Silver: 19.271 và 7.915 so với khoảng 27.100 các ngày khác)
  vì là 2 ngày ở đầu và cuối khoảng thời gian dữ liệu. Chi phí 2 ngày này thấp hơn do ít traffic, không phải do rẻ hơn.

Bảng Gold được Z-order theo `model` (8 file được viết lại, mỗi partition 1 file) để dashboard
lọc theo model đọc ít dữ liệu hơn.

**4. Vì sao tách 3 tầng**
- Mỗi tầng có một hợp đồng chất lượng riêng: thô / sạch / sẵn sàng cho báo cáo.
- Có thể chạy lại Silver/Gold từ Bronze mà không cần lấy lại dữ liệu từ nguồn.
- Người dùng cuối (dashboard) chỉ đọc 24 dòng Gold thay vì quét 190K dòng Silver.

Lưu ý: giá trong `COST_TABLE` là giá minh họa của lab, không phải giá thật.

Ảnh minh chứng: `submission/screenshots/nb04_bronze.png`, `nb04_silver.png`, `nb04_storage.png`,
`nb04_gold_build.png`, `nb04_gold_metrics.png`, `nb04_gold_checks.png`.